# 01 · ephys meta-org: inventory, recording book, protocols and checks

This notebook reads **every ABF file** under `data/`, matches it to the **recording book**, works out which
**protocol** was actually recorded and **checks** each file for problems, including **noise**. Nothing is changed or copied here:
the raw data are only read. Files are sorted into protocol folders later, in `02_ephys_anomaly-fix`, once
their problems are resolved.

**How to use:** run the cells from top to bottom. After every step there is a *review* cell; look at it before
moving on. Settings (data paths, thresholds, expected amplifier settings) are in `config.yaml` in this folder.

| Step | What happens | Review |
|---|---|---|
| 1 | read the header of every ABF | number of files, read errors |
| 2 | match each ABF to the recording book | unmatched / duplicate entries, book errors |
| 3 | classify the protocol from the ABF itself | protocol per file, disagreements with the book |
| 4 | noise: spectrum per recording (RMS, mains, pump peaks, artefacts) | spectra of flagged files |
| 5 | check against the expected setup | flags and their severity |
| 6 | save `recordings.csv`, `cells.csv`, `flags.csv`, `noise.csv` | originals unchanged |

## Setup
`FRESH = True` clears this subproject's `output/` folder first (derived files only; `data/` is never touched).
Close any output CSV that is open in Excel before running.

In [ ]:
FRESH = True

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
from ephys_analysis import meta_org
from ephys_analysis.utils import RunLog, find_repo_root, repo_path

REPO = find_repo_root()
cfg, out = meta_org.setup(REPO, fresh=FRESH)
log = RunLog(cfg, out)
print("ABF folder:    ", repo_path(cfg, cfg["paths"]["abf_root"]))
print("Recording book:", repo_path(cfg, cfg["paths"]["recording_book"]))

## Step 1 · Inventory
Every `.abf` below the ABF folder is opened read-only. For each file we keep the header information
(channels and units, sampling rate, sweeps, protocol name, tags, amplifier telegraph settings), a summary of
the command waveform, and a SHA-256 fingerprint used later to prove the file was not changed.

In [ ]:
recordings = meta_org.inventory(cfg, log=lambda msg: None)
print(f"{len(recordings)} ABF files, read errors: {(recordings['read_status'] != 'ok').sum()}")

In [ ]:
# Review: files per recording-day folder, and the channel set-up found
recordings["folder"] = recordings["relative_path"].str.rsplit("/", n=1).str[0]
display(recordings.groupby("folder").size().rename("n_files").to_frame())
display(recordings[["abf_filename", "abf_protocol", "adc_names", "adc_units", "dac_units", "sweep_count", "sweep_length_s"]])

## Step 2 · Recording book
Each ABF is matched to the *Recording Log* by its file name (`Recording Name`). A match is only accepted when it
is unique; duplicates are flagged, never guessed. The `Cell Tag` comes only from the recording book.

In [ ]:
recordings, book_issues, unmatched = meta_org.match_book(recordings, cfg)
display(recordings["match_status"].value_counts().to_frame())
print(f"Recording-book rows without an ABF here: {len(unmatched)}")

In [ ]:
# Review: anything not matched, and errors in the recording book itself
display(recordings.loc[recordings["match_status"] != "matched", ["abf_filename", "match_status", "match_note"]])
display(book_issues)

## Step 3 · Protocol classification
The **ABF decides** the protocol: the clamp mode comes from the channel units (pA = voltage clamp,
mV = current clamp) and the family from the command waveform (pulse, steps, ramp, flat). Only then is the
recording-book label compared: `confirmed` (agree), `probable` (no label), `mismatch`, `ambiguous`, `unsupported`.

In [ ]:
classification = meta_org.classify(recordings, cfg)
display(pd.crosstab(classification["abf_family"].replace("", "(none)"), classification["classification_status"]))

In [ ]:
# Review: recordings where the ABF and the recording book disagree or the protocol is unclear
display(classification.loc[classification["classification_status"] != "confirmed",
                           ["abf_filename", "label", "abf_family", "classification_status", "review_reason"]])

## Step 4 · Noise (spectral QC)
Each recording's **power spectrum** is computed from its quiet stretches only (command at holding, away from
command changes; in current clamp with spikes removed). Measured: **RMS noise** (1–1000 Hz), **mains** (50 Hz and
harmonics, peak-to-peak), narrow **low-frequency peaks** (0.05–20 Hz: perfusion / pump; only in recordings without
current injection) and **brief transient artefacts**. Below the `review` thresholds in `config.yaml` → `noise`
the flag is `info`; above, 02 asks for a decision: `accept`, `remove_mains` (sinusoid subtraction), `notch` (a
narrow notch at a given frequency) or `exclude`. Nothing is filtered here. In current clamp the RMS includes real
synaptic activity, so it is an upper bound of the recording noise.

In [ ]:
noise_tab, noise_flags = meta_org.noise(recordings, classification, cfg, out, log=lambda msg: None)
cols = ["abf_filename", "clamp_mode", "unit", "quiet_s", "rms", "mains_pp", "mains_50Hz_pp", "mains_150Hz_pp",
        "lowf_peaks", "artefacts_per_min"]
display(noise_tab[[c for c in cols if c in noise_tab]].round(3))
display(noise_flags[noise_flags["flag"] != "noise_not_assessed"][["abf_filename", "flag", "severity", "detail"]])

In [ ]:
# Review: the spectrum of every recording with a review-level noise flag (all spectra: output/noise_spectra/)
from IPython.display import Image
for rid in noise_flags.loc[noise_flags["severity"] == "review", "recording_id"].unique():
    f = noise_tab.loc[noise_tab["recording_id"] == rid, "figure"].iloc[0]
    if f:
        display(Image(filename=f, width=800))

## Step 5 · Checks against the expected setup
Every file is compared with what the data **should** look like (see `config.yaml` → `expected`):

* **Recording book:** matched? Cell Tag present? Date matches the ABF? Does a comment mention a current
  set on the amplifier (e.g. "-10pA holding"), which the ABF cannot show?
* **ABF header (MultiClamp 700B telegraph):** amplifier mode agrees with the channel units? Telegraph on?
  Command sensitivity as expected (400 pA/V current clamp, 20 mV/V voltage clamp)? Baseline protocols at 0 pA?
* **Signals:** current clamp with the voltage recorded twice and no current channel? Voltage outside the
  physiological range (gain error, e.g. ~20x)?

Each flag has a severity: **needs_fix** (must be corrected before analysis), **review** (a decision is
needed) or **info**. A recording's status is its worst flag.

In [ ]:
table, flags = meta_org.check(recordings, classification, book_issues, unmatched, cfg, log=lambda msg: None,
                              noise_flags=noise_flags)
display(table["status"].value_counts().to_frame())
display(flags.groupby(["severity", "flag"]).size().rename("n").to_frame())

In [ ]:
# Review: every needs_fix / review flag with its reason
display(flags.loc[flags["severity"] != "info", ["abf_filename", "flag", "severity", "detail"]])

## Step 6 · Save
Writes `output/recordings.csv` (one row per ABF), `output/cells.csv` (one row per Cell Tag) , `output/flags.csv` and `output/noise.csv`, then re-hashes every original ABF to confirm nothing was changed.

In [ ]:
paths = meta_org.save(table, flags, out, log.run_id, noise_tab)
changed = meta_org.originals_unchanged(table)
print("Originals unchanged (SHA-256)." if not changed else f"WARNING: changed originals: {changed}")
log.finish(paths)
for p in paths:
    print("written:", p)

**Next:** open `02_ephys_anomaly-fix/anomaly-fix.ipynb`.